In [1]:
import os
os.chdir('/workspace/321f0eeb-097c-4d58-9349-fa9ad588875f')
print(os.listdir('.'))


['brackets_0_2500.pkl', 'lmfdb_Lchi5.json', 'memory', '.kernel_llm_logs_1.txt', '.prompts', '.config', 'lchi5_module.py', '-PROMPT-v6-DATASET.md']


In [2]:
import pickle, json
with open('brackets_0_2500.pkl','rb') as f:
 brackets = pickle.load(f)
print(type(brackets), len(brackets))
print(brackets[0], brackets[-1])


<class 'list'> 2622
(6.5999999999999925, 6.699999999999992) (2499.399999999955, 2499.499999999955)


In [3]:
with open('lmfdb_Lchi5.json') as f:
 lmfdb = json.load(f)
print(type(lmfdb))
if isinstance(lmfdb, dict):
 print(list(lmfdb.keys())[:20])


JSONDecodeError: Expecting value: line 2 column 1 (char 1)

In [4]:
with open('lmfdb_Lchi5.json') as f:
 head = f.read(2000)
print(head[:1500])



# Zeros of 1-5-5.4-r0-0-0 downloaded from the LMFDB on 30 May 2026.
{"order_of_vanishing": 0, "positive_zeros": ["6.64845334472771471612327845997", "9.831444432886669616348321347458", "11.95884562608351453026565868826", "16.03382112838423567459325378224", "17.566994292325555202701595268144", "19.54073262278475025037869002299", "22.227405454459410911877624963081", "24.58846621740819520765626997608", "26.77609594800414011652357496527", "28.46103510017752247518697827232", "29.70790935048096556923098651865", "33.00045600687051436794975917721", "34.728812978904808674143729833981", "35.86863837181227459459504863887", "38.12918472143653185015141827037", "39.56057294640318170505509505995", "41.84243854579169430850930688531", "44.03129006144169504470090805842", "45.4273000827822893888415619096", "46.49272715949140534533919935167", "48.34566182106784617654820130449", "51.08775192674649135525825720413", "52.1259022313169741886041306959", "53.83044519544216335105426902233", "55.589280335404810150

In [5]:
with open('lmfdb_Lchi5.json') as f:
 lines = f.readlines()
# Find the JSON line
import json
data_line = [l for l in lines if l.strip().startswith('{')][0]
lmfdb = json.loads(data_line)
print(list(lmfdb.keys()))
pos = lmfdb['positive_zeros']
print(len(pos), pos[0], pos[128])


['order_of_vanishing', 'positive_zeros', 'negative_zeros', 'positive_zeros_accuracy', 'negative_zeros_accuracy']
129 6.64845334472771471612327845997 199.09102170624125905531999800239


In [6]:
import multiprocessing as mp_proc
print(mp_proc.cpu_count())


8


In [7]:
# Quick density check: density of zeros for Dirichlet L(chi, mod 5) is N(T) ~ (T/(2pi)) * log(5T/(2pi e)) approx
# At T~2500 we have 2622 zeros, so density ~ 1 per unit at large T. Let's estimate height needed for 5000 zeros.
import math
def N_approx(T):
 return (T/(2*math.pi)) * math.log(5*T/(2*math.pi*math.e))

for T in [2500, 3500, 4000, 4200, 4500, 5000]:
 print(T, N_approx(T))


2500 2624.3085849085446
3500 3861.461287722044
4000 4498.107339316029
4200 4755.626528649516
4500 5144.726642247403
5000 5800.206170198718


In [8]:
# N_approx(4500) ≈ 5145; that's enough. We need to scan from t=2500 to ~4500 in step h=0.1 (since brackets use 0.1 step from existing file).
# Let's confirm step
# Brackets start at 6.6/6.7 with step 0.1. So h=0.1
# Number of t evaluations: (4500-2500)/0.1 = 20000 per worker if single.
# We have 8 cores. Let's measure cost of Z(t) at high t.
import mpmath as mp
import time
mp.mp.dps = 15
from lchi5_module import Z
t0 = time.time()
for _ in range(20):
 v = Z(4400.0 + _ * 0.5)
print(20/(time.time()-t0), "evals/s @ dps=15 t~4400")


2.1617605928835246 evals/s @ dps=15 t~4400


In [9]:
# ~2 evals/s -> 20000 evals at high t is 10000s single-core; with 8 cores ~1250s. Manageable.
# Let's also check at t~2500
t0 = time.time()
for _ in range(20):
 v = Z(2500.0 + _ * 0.5)
print(20/(time.time()-t0), "evals/s @ dps=15 t~2500")


3.995827644645574 evals/s @ dps=15 t~2500


In [10]:
# Refinement cost at dps=50: estimate
mp.mp.dps = 55
t0 = time.time()
for _ in range(10):
 v = Z(4400.0 + _ * 0.5)
print(10/(time.time()-t0), "evals/s @ dps=55 t~4400")


1.8442864900636045 evals/s @ dps=55 t~4400


In [11]:
# Refinement uses anderson/bisection - typically ~20-50 evals/zero. 5000 zeros * 30 evals = 150000 evals at high dps
# At ~2 ev/s -> 75000s single-core, /8 cores = ~9400s. That's >2h. Too slow if budget is 3600s.
# We have 3600s total budget. Need to be careful.
# Let's plan:
# - Scan from 2500 to ~4500: estimate ~1500s with 8 cores
# - Refinement: this is the bottleneck. We could refine in parallel but ~9400s is too much.
# 
# Actually findroot with anderson should converge in ~6-8 iterations once bracket is tight (h=0.1 bracket).
# Let's test refinement speed for a single bracket at dps=50.
from lchi5_module import refine_bracket
t0 = time.time()
r = refine_bracket((6.6, 6.7, 50))
print(time.time()-t0, "s for one bracket dps=50 at low t")
print(r)


0.2597839832305908 s for one bracket dps=50 at low t
6.648453344727714716123278459979317847298585423244498


In [12]:
# Compare with LMFDB: 6.64845334472771471612327845997 matches.
t0 = time.time()
r = refine_bracket((2499.4, 2499.5, 50))
print(time.time()-t0, "s for one bracket dps=50 at t~2500")
print(r)

t0 = time.time()
r = refine_bracket((4400.0, 4400.1, 50))
print(time.time()-t0, "s for one bracket dps=50 at t~4400")


3.6642093658447266 s for one bracket dps=50 at t~2500
2499.495798958695817304372584361642586145710317981921


11.683537006378174 s for one bracket dps=50 at t~4400


In [13]:
# 11.7s per bracket at high t is too slow. We need a different approach.
# Let's reduce dps overhead in refine. The function uses target_dps+5 for working dps. Try anderson which evaluates a lot.
# Try Brent's method (default mp.findroot with two starting points might not use anderson efficiently).
# Actually, "anderson" solver uses ~5-6 iterations from a bracket usually. With dps=55 evals at ~1.8/s, ~3s expected.
# 11.7s suggests ~20 evals.
# Let's try a smarter approach: 
# 1. First refine at dps=15 to get a tight initial guess
# 2. Then polish to dps=50 with newton/secant
import time
mp.mp.dps = 15
t0 = time.time()
root_lo = mp.findroot(Z, (mp.mpf(4400.0), mp.mpf(4400.1)), solver='anderson')
print(time.time()-t0, "s low dps refinement")
print(root_lo)


5.5443127155303955 s low dps refinement
4399.6365570633


In [14]:
# Hmm 5.5s at dps=15 - that's because anderson with bracket might do many evals. The Z function at dps=15 evaluates at ~2.2/s
# So ~12 evals. Let's count.
# At dps=55 each eval is ~0.55s, so 20 evals = 11s. Matches.
# Try Newton's method at dps=55 with derivative.. but no derivative supplied.
# Alternative: use 'secant' or 'bisect'.
# Try a two-stage: first localize at dps=20, then refine at dps=55 with two close starting points (secant)
mp.mp.dps = 20
t0 = time.time()
root_lo = mp.findroot(Z, (mp.mpf(4400.0), mp.mpf(4400.1)), solver='anderson')
print(time.time()-t0, "s dps=20")
print(root_lo)


5.6566689014434814 s dps=20
4399.6365570633046505


In [15]:
# Now polish at dps=55 starting from (root_lo - 1e-10, root_lo + 1e-10) with secant
mp.mp.dps = 55
t0 = time.time()
r1 = mp.mpf(root_lo) - mp.mpf('1e-10')
r2 = mp.mpf(root_lo) + mp.mpf('1e-10')
root_hi = mp.findroot(Z, (r1, r2), solver='anderson')
print(time.time()-t0, "s polish")
print(mp.nstr(root_hi, 50))


7.1805994510650635 s polish
4399.6365570633046504559599633217802556092607425925


In [16]:
# Still 7s. anderson must do too many evals at high dps with bracketed start.
# Let me try with single starting point (Newton needs derivative; secant uses two close points).
mp.mp.dps = 55
t0 = time.time()
root_hi2 = mp.findroot(Z, mp.mpf(root_lo), solver='secant')
print(time.time()-t0, "s secant single")
print(mp.nstr(root_hi2, 50))


5.767785310745239 s secant single
4399.6365570633046504559599633217802556092607425925


In [17]:
# Let's try using the dps=15 root and only doing a few Newton iterations manually with numerical derivative
# But the key cost is each Z evaluation at dps=55 is ~0.55s. We need to minimize evals.
# Better: scan at finer step h=0.05 at dps=15 to get tighter brackets, then refine.
# But scanning at h=0.1 is fine.
# Key idea: do dps=15 refinement to get ~13 digits, then ONE Newton step at dps=55 (needs 2-3 evals).
# Actually anderson converges quadratically — should need only ~2 iterations after the dps=15 starting.

# Try with very tight bracket and bisect (which is monotone)
mp.mp.dps = 55
t0 = time.time()
# Use tight bracket around root_lo
a = mp.mpf(root_lo) - mp.mpf('1e-10')
b = mp.mpf(root_lo) + mp.mpf('1e-10')
za = Z(a); zb = Z(b)
print("za,zb signs:", za>0, zb>0, "time:", time.time()-t0)


za,zb signs: True False time: 1.1186280250549316


In [18]:
# Each eval at dps=55 is ~0.55s. We need approach minimizing evals.
# Strategy: 
# Plan: Refine all 5000 brackets at dps=15 first (fast), then polish at dps=55 with secant from tight start.
# At dps=15: refine cost ~5s at high t, /8 cores = ~0.6s/zero -> 5000*0.6 = 3000s. Too slow!
#
# Need to reduce dps=15 refinement cost. Anderson takes too many evals.
# Try Brent: solver='ridder' or 'bisect'

mp.mp.dps = 15
t0 = time.time()
r = mp.findroot(Z, (mp.mpf(4400.0), mp.mpf(4400.1)), solver='bisect')
print(time.time()-t0, "s bisect dps=15")

mp.mp.dps = 15
t0 = time.time()
r = mp.findroot(Z, (mp.mpf(4400.0), mp.mpf(4400.1)), solver='ridder')
print(time.time()-t0, "s ridder dps=15")


ValueError: Could not find root within given tolerance. (25.6289943435603519573 > 2.16840434497100886801e-19)
Try another starting point or tweak arguments.

In [19]:
# anderson is fine. The cost issue is fundamental: Z(t) at high t and high dps is expensive.
# Real bottleneck check - mp.dirichlet computation
# Let's check parallelization feasibility:
# Refinement at dps=55, 5000 zeros, with ~7-12s per zero at high t, /8 cores = 4400-7500s. Over budget.
# 
# Alternative: lower target dps. The spec says dps=50 for high precision but we have ~3600s total.
# Let me first do the scan, then decide.
# Actually let's check if N(T) at 4500 covers exactly 5000. We need 5000 zeros total.
# So we need t up to ~4470 or so.
# Start the scan now in parallel.
import os, pickle
print(os.cpu_count())


8


In [20]:
# Setup the parallel scan from 2500 to 4500
# Use 8 workers, split interval into 8 chunks
# But: high-t evaluations are slower, so should distribute as round-robin or similar load balancing.
# For simplicity split into more, smaller chunks (e.g., 80 chunks of width 25) and use Pool.imap_unordered.

# Write a runner script
scan_script = r'''
import sys, os, time, pickle
sys.path.insert(0, '/workspace/321f0eeb-097c-4d58-9349-fa9ad588875f')
os.chdir('/workspace/321f0eeb-097c-4d58-9349-fa9ad588875f')
from lchi5_module import scan_interval
from multiprocessing import Pool

T_START = 2499.4 # start slightly before last bracket end to avoid missing
T_END = 4500.0
H = 0.1
CHUNK = 25.0 # width of each task
DPS = 15

tasks = []
t = T_START
while t < T_END:
 tasks.append((t, min(t+CHUNK, T_END), H, DPS))
 t += CHUNK

print(f"Created {len(tasks)} tasks; total interval {T_START}-{T_END}", flush=True)

t0 = time.time()
all_brackets = []
with Pool(8) as p:
 for i, res in enumerate(p.imap_unordered(scan_interval, tasks)):
 all_brackets.extend(res)
 elapsed = time.time()-t0
 print(f"[{elapsed:7.1f}s] done {i+1}/{len(tasks)} (+{len(res)} brackets, total {len(all_brackets)})", flush=True)
 # Save partial result
 with open('brackets_2500_4500_partial.pkl', 'wb') as f:
 pickle.dump(all_brackets, f)

# Sort and dedupe by midpoint
all_brackets.sort(key=lambda ab: ab[0])
with open('brackets_2500_4500.pkl', 'wb') as f:
 pickle.dump(all_brackets, f)
print("Done in", time.time()-t0, "s; total new brackets:", len(all_brackets))
'''

with open('run_scan.py', 'w') as f:
 f.write(scan_script)
print("written")


written


In [21]:
# Launch the scan in background; we'll monitor.
import subprocess, time
log = open('scan.log', 'w')
proc = subprocess.Popen(['python', 'run_scan.py'], stdout=log, stderr=subprocess.STDOUT)
print("PID:", proc.pid)
time.sleep(8)
with open('scan.log') as f:
 print(f.read())


PID: 48


Created 81 tasks; total interval 2499.4-4500.0



In [22]:
# Monitor scan. While we wait, plan refinement strategy.
# Test: At dps=20, evaluating Z(t)...
import mpmath as mp2
from lchi5_module import Z as Z_mod # same
# the worker process owns its own mp.mp.dps. We don't conflict here.
# Strategy: refine each bracket to dps=50 via 2-stage:
# stage 1: dps=15 with anderson (gives ~12 digits)
# stage 2: dps=55 with secant starting from (root_lo - eps, root_lo + eps) where eps=1e-12
# Actually anderson does many evals. Try with mp.findroot ... using single point and 'newton'? newton needs derivative.
# Let me try 'secant' with tight initial gap at dps=55 directly.

mp2.mp.dps = 55
t0 = time.time()
# Use root estimate from dps=15 stage
r_est = mp2.mpf('4399.6365570633') # 12 digits
r = mp2.findroot(Z_mod, (r_est - mp2.mpf('1e-10'), r_est + mp2.mpf('1e-10')), solver='secant', tol=mp2.mpf('1e-50'))
print(time.time()-t0, "s secant tight")
print(mp2.nstr(r, 55))


11.583354234695435 s secant tight
4399.636557063304650455959963321780255609260742592481923


In [23]:
# Still 11s. The cost is dominated by Z evals at dps=55. Secant from 1e-10 separation converges quadratically:
# 10 -> 20 -> 40 -> 80 digits in ~3 iterations, but each iteration has 1-2 evals plus extra overhead.
# 11s = ~20 evals * 0.55s each. mpmath findroot might be doing more iterations than needed.

# Let me try newton with derivative computed by mp.diff
def Z_with_diff(t):
 return Z_mod(t)

mp2.mp.dps = 55
t0 = time.time()
# direct mp.findroot with single starting point and 'newton' tries numerical derivative
r = mp2.findroot(Z_mod, mp2.mpf('4399.6365570633'), tol=mp2.mpf('1e-50'))
print(time.time()-t0, "s default")
print(mp2.nstr(r, 55))


14.522737503051758 s default
4399.636557063304650455959963321780255609260742592481923


In [24]:
# Let me see: maybe Z evaluation at dps=55 isn't as slow if we lower precision internally with reduced dirichlet term count.
# But Z is given by module. Look at mp.dirichlet - this is exact (finite sum).
# CHI mod 5 has period 5, so dirichlet sum is L(s, chi) = 5^(-s) * sum_{k=1..4} chi(k) * zeta(s, k/5) - so mp.dirichlet uses Hurwitz zeta.

# Time test of mp.dirichlet at dps=55, t=4400
mp2.mp.dps = 55
t0 = time.time()
s = mp2.mpc(mp2.mpf('0.5'), mp2.mpf('4400'))
v = mp2.dirichlet(s, [0,1,-1,-1,1])
print(time.time()-t0, "s for one dirichlet eval at dps=55, t=4400")
# At dps=15:
mp2.mp.dps = 15
t0 = time.time()
s = mp2.mpc(mp2.mpf('0.5'), mp2.mpf('4400'))
v = mp2.dirichlet(s, [0,1,-1,-1,1])
print(time.time()-t0, "s at dps=15")


1.5452947616577148 s for one dirichlet eval at dps=55, t=4400


1.0693016052246094 s at dps=15


In [25]:
# Surprising — 1s per eval at dps=15! That's worse than expected. The first calls might cache.
# Try again
mp2.mp.dps = 15
t0 = time.time()
for i in range(5):
 v = mp2.dirichlet(mp2.mpc(mp2.mpf('0.5'), mp2.mpf(4400+i*0.1)), [0,1,-1,-1,1])
print((time.time()-t0)/5, "s/eval dps=15 t=4400 (warm)")

mp2.mp.dps = 55
t0 = time.time()
for i in range(5):
 v = mp2.dirichlet(mp2.mpc(mp2.mpf('0.5'), mp2.mpf(4400+i*0.1)), [0,1,-1,-1,1])
print((time.time()-t0)/5, "s/eval dps=55 t=4400 (warm)")


1.1365049839019776 s/eval dps=15 t=4400 (warm)


1.2021169185638427 s/eval dps=55 t=4400 (warm)


In [26]:
# Hmm Z eval has Z = (mp.exp(1j*theta) * L).real - theta evaluation includes gamma which is also expensive
# So Z eval at dps=15 is ~1.1-1.2s at t=4400.
# 20000 Z evals at t=4400 takes 24000s single, /8 = 3000s — matches earlier estimate.
# 
# But our scan loop runs ~at this speed. Total scan work: integral over [2500, 4500] of cost-per-eval / step.
# Cost per eval ~ proportional to log(t) or sqrt(t)? Hard to know but assume roughly constant 1s in average.
# 20000 evals * ~0.7s avg = 14000s, /8 = 1750s. Workable.

# Let me check the scan log:
with open('scan.log') as f:
 print(f.read())


Created 81 tasks; total interval 2499.4-4500.0



In [27]:
import time
time.sleep(60)
with open('scan.log') as f:
 print(f.read()[-3000:])


Created 81 tasks; total interval 2499.4-4500.0
[ 123.3s] done 1/81 (+31 brackets, total 31)
[ 124.2s] done 2/81 (+31 brackets, total 62)
[ 124.7s] done 3/81 (+30 brackets, total 92)
[ 125.7s] done 4/81 (+31 brackets, total 123)
[ 125.8s] done 5/81 (+30 brackets, total 153)
[ 126.7s] done 6/81 (+30 brackets, total 183)
[ 127.5s] done 7/81 (+30 brackets, total 213)
[ 129.0s] done 8/81 (+30 brackets, total 243)



In [28]:
# Excellent: 8 chunks of 25 width = 200 units of t covered in 125s. Each chunk has 250 evals.
# Rate per chunk = ~125s/chunk; with 8 parallel = 250/125 = 2 evals per 1s effectively per worker
# Total task units: 81. With 8 parallel, expected wall-clock ~81/8 * 125s = 1265s.
# But later chunks are slower (higher t). Let's estimate more carefully.

# Plan refinement timing:
# At dps=15 anderson with bracket of width 0.1: typically converges in ~12 evals (1 evals/iter, anderson is broyden-like)
# At t~4000-4500, eval ~1.1s -> 12 evals = 13s per zero
# 5000 zeros * 13s / 8 cores = 8125s. Too slow.
# 
# We have ~2400s total budget remaining (3600 - 1200 scan).
# Need a much faster refinement.
# 
# Key observation: For high-t Z evaluation, mp.dirichlet eval time depends on Im(s) (uses Riemann-Siegel-like algorithm? actually Euler-Maclaurin)
# - this is the bottleneck regardless of refinement scheme.
# 
# Strategy: do refinement at dps=15 ONLY (gives ~12-13 digits which is fine for verification), then for the FIRST 129 zeros do dps=50 polishing.
# But the spec requires dps=50 zeros.
# 
# Compromise: refine ALL zeros at dps=15 first (which validates against LMFDB to 12 digits) and store both:
# - dps15 refined values (~5000 zeros)
# - dps50 polished values for first 129 (to verify match LMFDB to 28 digits)
# 
# Actually the task is to "polish each zero to dps=50". We may not have time.
# Let me consider: with dps=15 first, then for each zero do one Newton step at dps=55 with derivative computed by mp.diff (1 extra eval at dps=55).
# But mp.diff would evaluate Z at perturbed points... maybe 4-5 evals.

# Let's first commit to: refine all zeros at dps=15 (rough), check timing, then decide.
# Estimate: 5000 zeros * 12 evals/zero / 8 cores * 1s/eval (avg) = 7500s. Still too slow.
# 
# We need to reduce evals per zero. Use bisection-with-acceleration:
# 1. We have bracket (a,b) of width 0.1 where Z(a) and Z(b) have opposite signs
# 2. One secant step: t_new = a - Z(a)*(b-a)/(Z(b)-Z(a)) -- 0 evals (already have Z(a), Z(b))
# 3. Eval Z(t_new), assign new bracket -- 1 eval
# 4. Repeat: typically 4-5 iterations gives 10+ digits since secant is superlinear
# 
# But we need Z(a) and Z(b) already from the SCAN. The scan saves brackets only as (t_start, t_end) pairs, not Z values.
# We'd need to re-evaluate. But Z at scan points are at dps=15 (low precision).
# 
# Better: do the refinement using values from the scan. Modify the scan to also save Z values? Not now.
# 
# Alternative: Use anderson's solver more efficiently. Or just commit to a faster heuristic.

# Let's just go with dps=15 refinement (anderson) for all 5000, and dps=55 polish for first 129.
# 5000 * ~12s wall at high-t average — let me check anderson eval count carefully.
import mpmath as mp3
from lchi5_module import Z
mp3.mp.dps = 15
counter = [0]
def Z_count(t):
 counter[0] += 1
 return Z(t)

t0 = time.time()
r = mp3.findroot(Z_count, (mp3.mpf(2499.4), mp3.mpf(2499.5)), solver='anderson')
print(time.time()-t0, "s evals:", counter[0])


8.977809190750122 s evals: 11


In [29]:
# 11 evals at dps=15, t~2500 = 9s. So 5000 * 9s = 45000s single core, /8 = 5600s. Too slow.
# 
# Custom fast refinement: use secant iteration manually with fewer evals
def fast_refine(a, b, target_dps=15, tol=mp3.mpf('1e-14')):
 mp3.mp.dps = target_dps + 2
 a = mp3.mpf(a); b = mp3.mpf(b)
 za = Z(a); zb = Z(b)
 for _ in range(20):
 if za == zb:
 return (a+b)/2
 # secant
 c = a - za*(b-a)/(zb-za)
 if c <= a or c >= b: # outside bracket, fallback bisection
 c = (a+b)/2
 zc = Z(c)
 if abs(zc) < tol or abs(b-a) < tol:
 return c
 if (zc > 0) == (za > 0):
 a, za = c, zc
 else:
 b, zb = c, zc
 return c

counter[0] = 0
mp3.mp.dps = 15
t0 = time.time()
# Custom version with counter
def fast_refine_count(a, b, target_dps=15, tol=mp3.mpf('1e-14'), maxiter=15):
 mp3.mp.dps = target_dps + 2
 a = mp3.mpf(a); b = mp3.mpf(b)
 za = Z(a); zb = Z(b); counter[0] += 2
 for _ in range(maxiter):
 if za == zb: return (a+b)/2
 c = a - za*(b-a)/(zb-za)
 if c <= a or c >= b: c = (a+b)/2
 zc = Z(c); counter[0] += 1
 if abs(zc) < tol or abs(b-a) < tol: return c
 if (zc > 0) == (za > 0):
 a, za = c, zc
 else:
 b, zb = c, zc
 return c

r = fast_refine_count(2499.4, 2499.5)
print(time.time()-t0, "s, evals:", counter[0], "root:", r)


3.9299192428588867 s, evals: 7 root: 2499.4957989586958


In [30]:
# 7 evals at dps~17, 4s at t=2500. Good, halves the cost vs anderson.
# Try with t=4400:
counter[0] = 0
t0 = time.time()
r = fast_refine_count(4399.6, 4399.7)
print(time.time()-t0, "s, evals:", counter[0], "root:", r)


16.562148094177246 s, evals: 14 root: 4399.6365570633047


In [31]:
# 14 evals at high t, because secant convergence may be slow when bracket bigger. Let me check Z(a),Z(b) on initial bracket.
# Actually the issue: bracket (4399.6, 4399.7) — let me check what scan would actually return.
# Look at the scan: it had stepped by 0.1 from 2500 — but the brackets we see start at 6.6, 6.7 (which is 0.1 step from 2500-2493.4=6.6? No, the first bracket is at low t).
# Anyway, brackets are 0.1 wide. Width 0.1 at t=4400 — Z'(t) is large, ~5-15. So Z spans ~1 over the bracket.
# Secant should converge fast. 14 evals seems high.

# Different idea: use fewer iterations with looser tol
def fast_refine_v2(a, b, target_dps=15, maxiter=8):
 mp3.mp.dps = target_dps + 5
 a = mp3.mpf(a); b = mp3.mpf(b)
 za = Z(a); zb = Z(b); counter[0] += 2
 eps = mp3.mpf(10)**(-target_dps)
 for i in range(maxiter):
 c = a - za*(b-a)/(zb-za) if zb != za else (a+b)/2
 if not (a < c < b): c = (a+b)/2
 if (b - a) < eps * abs(c): return c
 zc = Z(c); counter[0] += 1
 # update bracket
 if (zc > 0) == (za > 0):
 a, za = c, zc
 else:
 b, zb = c, zc
 return c

counter[0]=0
t0 = time.time()
r = fast_refine_v2(4399.6, 4399.7, target_dps=14, maxiter=6)
print(time.time()-t0, "s evals:", counter[0], "root:", r)


9.515095472335815 s evals: 8 root: 4399.636557059891027


In [32]:
# 8 evals, ~9.5s at t=4400. Get's ~10 digits which is good enough for stage 1.
# 5000 * 8 evals * 1.0s avg / 8 cores = 5000s. Still bad.
# 
# Let me step back. 3600s total budget, and the scan is taking ~1500s.
# Realistically we may not be able to fully refine all 5000 zeros to dps=50.
# 
# Honest plan: 
# 1. Complete the scan (running now ~1200s estimated)
# 2. Refine all brackets to dps=15 (Stage A: gives ~10-13 digit zeros) - this is the dominant cost
# 3. For first 129 zeros, polish to dps=50 and verify against LMFDB
# 
# This satisfies most of the objective except "polish each zero to dps=50". 
# We must be honest about the limitation.
# 
# Alternative: skip refining the OLD 2622 brackets if we have their data? No — we don't have refined zeros.
# 
# Actually wait — re-examine: report r13 saved brackets but apparently did not refine. So we must refine all 5000 from scratch.
# 
# Let me check whether mp.dirichlet is actually the bottleneck or it's something else.

# Profile single eval
import cProfile, pstats, io
mp3.mp.dps = 20
pr = cProfile.Profile()
pr.enable()
for _ in range(3):
 v = Z(mp3.mpf(4400.0))
pr.disable()
s = io.StringIO()
ps = pstats.Stats(pr, stream=s).sort_stats('cumulative')
ps.print_stats(15)
print(s.getvalue())


 2247100 function calls (2247096 primitive calls) in 5.470 seconds

 Ordered by: cumulative time
 List reduced from 214 to 15 due to restriction <15>

 ncalls tottime percall cumtime percall filename:lineno(function)
 3/2 0.000 0.000 5.469 2.735 /app/kernel_env/lib/python3.13/site-packages/IPython/core/interactiveshell.py:3711(run_code)
 2 0.000 0.000 5.469 2.735 {built-in method builtins.exec}
 3 0.000 0.000 5.468 1.823 /workspace/321f0eeb-097c-4d58-9349-fa9ad588875f/lchi5_module.py:9(Z)
 12 0.000 0.000 5.462 0.455 /app/kernel_env/lib/python3.13/site-packages/mpmath/functions/zeta.py:528(zeta)
 12 0.004 0.000 5.460 0.455 /app/kernel_env/lib/python3.13/site-packages/mpmath/functions/zeta.py:582(_hurwitz)
 12 0.109 0.009 5.381 0.448 /app/kernel_env/lib/python3.13/site-packages/mpmath/functions/zeta.py:660(_hurwitz_em)
 3 0.000 0.000 4.438 1.479 /app/kernel_env/lib/python3.13/site-packages/mpmath/functions/zeta.py:787(dirichlet)
 84 0.008 0.000 4.230 0.050 /app/kernel_env/lib/python3.13/

In [33]:
# So Z eval at dps=20, t=4400 = 1.8s. Hurwitz_em dominates. The cost is in mp.hurwitz.
# 
# Let me also check the scan log:
with open('scan.log') as f:
 print(f.read()[-2000:])


Created 81 tasks; total interval 2499.4-4500.0
[ 123.3s] done 1/81 (+31 brackets, total 31)
[ 124.2s] done 2/81 (+31 brackets, total 62)
[ 124.7s] done 3/81 (+30 brackets, total 92)
[ 125.7s] done 4/81 (+31 brackets, total 123)
[ 125.8s] done 5/81 (+30 brackets, total 153)
[ 126.7s] done 6/81 (+30 brackets, total 183)
[ 127.5s] done 7/81 (+30 brackets, total 213)
[ 129.0s] done 8/81 (+30 brackets, total 243)
[ 247.1s] done 9/81 (+31 brackets, total 274)
[ 248.9s] done 10/81 (+31 brackets, total 305)
[ 249.0s] done 11/81 (+31 brackets, total 336)
[ 251.6s] done 12/81 (+30 brackets, total 366)
[ 251.6s] done 13/81 (+30 brackets, total 396)
[ 252.2s] done 14/81 (+31 brackets, total 427)
[ 252.5s] done 15/81 (+31 brackets, total 458)
[ 254.4s] done 16/81 (+31 brackets, total 489)



In [34]:
# Scan is progressing: 16/81 in 254s. Expected ~1280s total for scan.
# 
# Decision: We cannot afford full dps=50 refinement of all 5000 brackets in remaining ~2000s.
# 
# Let me think about this:
# - 5000 brackets * ~10s/refine at dps=50 high-t = 50,000s single-core = 6250s on 8 cores
# - Even dps=15 refinement of all is ~5000s/8 cores = 625s, doable, but precision insufficient
#
# Realistic deliverable: 
# - Provide refined zeros at dps=20-30, NOT dps=50
# - Or refine first ~1000 zeros to dps=50 (lower t range, fast)
#
# But research objective explicitly says polish each zero to dps=50.
# 
# Let me check: can scan also be run AT THE SAME TIME as refinement in parallel? But all 8 cores already in use.
# 
# We have constraint: 3600s budget, started ~5min ago. We have ~2700s left.
# 
# OPTION: Refine all zeros at dps=15 (cheap), then polish ALL at dps=55 -- but that's the expensive part.
# 
# Let me explore if Hurwitz can be computed faster. The hurwitz_em algorithm has cost roughly O(N) terms where N depends on dps and t.
# At dps=15, t=4400: ~23000 terms. That's the EM truncation point. EM error decreases as ((|t|+a)/N)^N.
# 
# To get 15 digits with EM: need N such that (t/N)^N ≈ 10^-15 => for t=4400, N~25 not bad. But hurwitz_em is being run multiple times (12 calls).
# Actually mp.dirichlet does L(s, chi) = q^(-s) * sum_{k} chi(k) * zeta(s, k/q) ... so calls hurwitz 4 times (chi values at 1,2,3,4).
# 
# Custom faster implementation possible via functional equation? Too complex.
#
# Let me decide: focus on the COMPLETE task in best-effort fashion.
# Stage 1: Run scan to ~T such that we have ~5000 brackets total
# Stage 2: Refine to dps=15 (which gives ~10 digits)
# Stage 3: Polish first 129 to dps=50 for LMFDB validation
# Stage 4: Document the dps limitation
#
# Continue monitoring scan.
import time
time.sleep(120)
with open('scan.log') as f:
 print(f.read()[-1500:])


Created 81 tasks; total interval 2499.4-4500.0
[ 123.3s] done 1/81 (+31 brackets, total 31)
[ 124.2s] done 2/81 (+31 brackets, total 62)
[ 124.7s] done 3/81 (+30 brackets, total 92)
[ 125.7s] done 4/81 (+31 brackets, total 123)
[ 125.8s] done 5/81 (+30 brackets, total 153)
[ 126.7s] done 6/81 (+30 brackets, total 183)
[ 127.5s] done 7/81 (+30 brackets, total 213)
[ 129.0s] done 8/81 (+30 brackets, total 243)
[ 247.1s] done 9/81 (+31 brackets, total 274)
[ 248.9s] done 10/81 (+31 brackets, total 305)
[ 249.0s] done 11/81 (+31 brackets, total 336)
[ 251.6s] done 12/81 (+30 brackets, total 366)
[ 251.6s] done 13/81 (+30 brackets, total 396)
[ 252.2s] done 14/81 (+31 brackets, total 427)
[ 252.5s] done 15/81 (+31 brackets, total 458)
[ 254.4s] done 16/81 (+31 brackets, total 489)
[ 369.3s] done 17/81 (+31 brackets, total 520)
[ 370.9s] done 18/81 (+32 brackets, total 552)
[ 371.6s] done 19/81 (+30 brackets, total 582)
[ 373.3s] done 20/81 (+30 brackets, total 612)
[ 374.5s] done 21/81 (+31

In [35]:
# Scan is taking ~125s per 8 chunks. 81 chunks total. Estimated total scan time: 81/8 * 125 = 1266s.
# But chunks at higher t will take longer. Conservative estimate: 1500-1800s total.
# 
# We have used ~6 min (360s of 3600). Scan ETA ~1500s. Remaining for refinement: ~1700s.
# 
# Math: scan started at ~125s mark, projecting total scan to ~1700s (28 min in).
# Refinement budget then = 3600 - 1700 = 1900s. 
# Refine 5000 zeros at dps=15 takes ~5000s/8 cores = 625s. Doable!
# Polish 129 zeros at dps=50 in low-t region: 129 * 3s = ~400s single-core, or ~50s parallel. Doable.
# Cache the result.
# 
# So feasible plan: dps=15 refinement only for full list, plus dps=50 for first ~500 zeros if time allows.
# We MUST be transparent that the full dps=50 refinement was not feasible within the time budget.
# 
# Continue waiting.
import time
time.sleep(240)
with open('scan.log') as f:
 content = f.read()
print(content[-2000:])


Created 81 tasks; total interval 2499.4-4500.0
[ 123.3s] done 1/81 (+31 brackets, total 31)
[ 124.2s] done 2/81 (+31 brackets, total 62)
[ 124.7s] done 3/81 (+30 brackets, total 92)
[ 125.7s] done 4/81 (+31 brackets, total 123)
[ 125.8s] done 5/81 (+30 brackets, total 153)
[ 126.7s] done 6/81 (+30 brackets, total 183)
[ 127.5s] done 7/81 (+30 brackets, total 213)
[ 129.0s] done 8/81 (+30 brackets, total 243)
[ 247.1s] done 9/81 (+31 brackets, total 274)
[ 248.9s] done 10/81 (+31 brackets, total 305)
[ 249.0s] done 11/81 (+31 brackets, total 336)
[ 251.6s] done 12/81 (+30 brackets, total 366)
[ 251.6s] done 13/81 (+30 brackets, total 396)
[ 252.2s] done 14/81 (+31 brackets, total 427)
[ 252.5s] done 15/81 (+31 brackets, total 458)
[ 254.4s] done 16/81 (+31 brackets, total 489)
[ 369.3s] done 17/81 (+31 brackets, total 520)
[ 370.9s] done 18/81 (+32 brackets, total 552)
[ 371.6s] done 19/81 (+30 brackets, total 582)
[ 373.3s] done 20/81 (+30 brackets, total 612)
[ 374.5s] done 21/81 (+31

In [36]:
# Great progress: 40/81 done at 619s. Scan ETA ≈ 81/40 * 619 = 1253s. 
# Each batch of 8 chunks takes ~120s, very consistent (less than-linear scaling with t).
# 
# Total scan should finish around 1250-1300s mark.
# 
# Meanwhile, prepare the refinement script. Use multiprocessing chunks.

refine_script = r'''
import sys, os, time, pickle
sys.path.insert(0, '/workspace/321f0eeb-097c-4d58-9349-fa9ad588875f')
os.chdir('/workspace/321f0eeb-097c-4d58-9349-fa9ad588875f')
import mpmath as mp
from lchi5_module import Z
from multiprocessing import Pool

def fast_refine_secant(args):
 """Refine bracket (a,b) to ~target_dps digits via secant iteration."""
 a, b, target_dps, maxiter = args
 mp.mp.dps = target_dps + 5
 a = mp.mpf(a); b = mp.mpf(b)
 za = Z(a); zb = Z(b)
 eps = mp.mpf(10)**(-target_dps)
 c = (a+b)/2
 for _ in range(maxiter):
 if zb == za:
 c = (a+b)/2
 else:
 c = a - za*(b-a)/(zb-za)
 if not (a < c < b):
 c = (a+b)/2
 if (b - a) < eps * max(abs(c), mp.mpf(1)):
 break
 zc = Z(c)
 if abs(zc) < eps:
 break
 if (zc > 0) == (za > 0):
 a, za = c, zc
 else:
 b, zb = c, zc
 return mp.nstr(c, target_dps+2, strip_zeros=False)

def refine_chunk(args):
 brackets, target_dps, maxiter = args
 mp.mp.dps = target_dps + 5
 out = []
 for (a, b) in brackets:
 s = fast_refine_secant((a, b, target_dps, maxiter))
 out.append(s)
 return out

# Load all brackets
with open('brackets_0_2500.pkl','rb') as f:
 b1 = pickle.load(f)
with open('brackets_2500_4500.pkl','rb') as f:
 b2 = pickle.load(f)

# Dedupe: remove overlapping near 2500 boundary
all_b = b1 + b2
all_b = list({(round(float(a),4), round(float(b),4)): (a,b) for a,b in all_b}.values())
all_b.sort(key=lambda ab: ab[0])
print(f"Total unique brackets: {len(all_b)}", flush=True)

TARGET_DPS = int(os.environ.get('TARGET_DPS', '15'))
MAXITER = int(os.environ.get('MAXITER', '12'))
CHUNK_SIZE = 50

# Build chunks: distribute by interleaving so each worker gets mix of low/high-t
chunks = []
chunks_count = (len(all_b) + CHUNK_SIZE - 1) // CHUNK_SIZE
for ci in range(chunks_count):
 bs = all_b[ci*CHUNK_SIZE:(ci+1)*CHUNK_SIZE]
 chunks.append((bs, TARGET_DPS, MAXITER))

print(f"Chunks: {len(chunks)}, target_dps={TARGET_DPS}, maxiter={MAXITER}", flush=True)

t0 = time.time()
results = [None]*len(chunks)
with Pool(8) as p:
 for i, res in enumerate(p.imap_unordered(refine_chunk, [(c[0],c[1],c[2]) for c in chunks])):
 # We don't know which chunk index this is via imap_unordered. Use enumerate-based saving.
 pass

# Need ordered map. Use imap (ordered).
t0 = time.time()
results = []
with Pool(8) as p:
 for i, res in enumerate(p.imap(refine_chunk, chunks)):
 results.append(res)
 elapsed = time.time()-t0
 flat = sum((len(r) for r in results), 0)
 print(f"[{elapsed:7.1f}s] chunk {i+1}/{len(chunks)} done; total refined: {flat}", flush=True)
 # save partial
 with open(f'zeros_partial_dps{TARGET_DPS}.pkl', 'wb') as f:
 pickle.dump([z for chunk in results for z in chunk], f)

flat_results = [z for chunk in results for z in chunk]
print(f"Total refined: {len(flat_results)} in {time.time()-t0:.1f}s")
# Sort numerically
flat_results.sort(key=lambda s: mp.mpf(s))
with open(f'zeros_Lchi5_N{len(flat_results)}_dps{TARGET_DPS}.pkl', 'wb') as f:
 pickle.dump(flat_results, f)
print(f"Saved zeros_Lchi5_N{len(flat_results)}_dps{TARGET_DPS}.pkl")
'''

with open('run_refine.py', 'w') as f:
 f.write(refine_script)
print("written")


written


In [37]:
import time
time.sleep(300)
with open('scan.log') as f:
 print(f.read()[-2000:])


s, total 274)
[ 248.9s] done 10/81 (+31 brackets, total 305)
[ 249.0s] done 11/81 (+31 brackets, total 336)
[ 251.6s] done 12/81 (+30 brackets, total 366)
[ 251.6s] done 13/81 (+30 brackets, total 396)
[ 252.2s] done 14/81 (+31 brackets, total 427)
[ 252.5s] done 15/81 (+31 brackets, total 458)
[ 254.4s] done 16/81 (+31 brackets, total 489)
[ 369.3s] done 17/81 (+31 brackets, total 520)
[ 370.9s] done 18/81 (+32 brackets, total 552)
[ 371.6s] done 19/81 (+30 brackets, total 582)
[ 373.3s] done 20/81 (+30 brackets, total 612)
[ 374.5s] done 21/81 (+31 brackets, total 643)
[ 374.7s] done 22/81 (+31 brackets, total 674)
[ 375.3s] done 23/81 (+32 brackets, total 706)
[ 377.0s] done 24/81 (+30 brackets, total 736)
[ 490.5s] done 25/81 (+31 brackets, total 767)
[ 490.9s] done 26/81 (+32 brackets, total 799)
[ 491.4s] done 27/81 (+31 brackets, total 830)
[ 494.4s] done 28/81 (+31 brackets, total 861)
[ 494.4s] done 29/81 (+31 brackets, total 892)
[ 495.5s] done 30/81 (+31 brackets, total 923)

In [38]:
# 50/81 at 976s. Linear extrapolation: 81/50 * 976 = 1581s. So ~600s more.
# That puts us at ~1581s, leaving ~2020s for refinement.
# 
# At dps=15 refinement: estimated 5000 * ~6s avg = 30000s single / 8 cores = 3750s. Too long.
# Need to reduce maxiter. With h=0.1 bracket and 4 secant iters, we get ~10-12 digits (secant is superlinear).
# Let's verify: 4 iters of secant from a width-0.1 bracket... error halves quadratically: 0.1 -> 0.001 -> 0.0001 (3 iters at superlinear order phi=1.618).
# Actually error after n secant iters: e_n ~ e_0^(phi^n). For e_0=0.1 and 4 iters: 0.1^6.85 = 1.4e-7. Not 15 digits.
# Need ~7-8 iters to get 15 digits.
# 
# But realistically the FIRST eval at the midpoint reveals a much tighter bracket. And once we're close, convergence is very fast.
# 
# Hmm. Let me reconsider. We could LOWER target_dps to ~12, which is still meaningful precision (matches LMFDB to 12 digits, while LMFDB gives 28-29).
# With target_dps=12 and maxiter=6, refinement per zero at high t = 6 evals * 1.0s = 6s.
# 5000 * 6s / 8 cores = 3750s. Still too slow.
# 
# More aggressive: target_dps=10, maxiter=5: 5 evals * 1.0s = 5s. 5000*5/8 = 3125s. STILL too slow.
# 
# Bottleneck is per-eval cost at high t (~1s). To get under 1000s wall total: 1000*8 = 8000s CPU = 8000 evals total max.
# That's only 1.6 evals/zero average. Impossible.
# 
# So full refinement of 5000 zeros within 2000s wall is infeasible.
# 
# Realistic deliverable: refine first ~1500-2000 zeros at dps=15-50.
# Or use 1 secant iteration only (gives ~5-6 digits, just as a quick estimate).
# 
# Strategy: I'll dedicate refinement time as follows:
# Phase A (~1000s): Refine all 5000 brackets with target_dps=12, maxiter=5 — gives 8-10 digit zeros
# Phase B (~500s): For low-t portion (first ~1000 zeros, t < 1000), polish to dps=50
# 
# This still might exceed budget. Let me be pragmatic and start refinement on what we have NOW for the 2622 already done brackets, in parallel with scan.
# 
# But scan uses all 8 cores currently. Bad idea to compete.
# 
# Wait until scan finishes. Continue monitoring.
import time
time.sleep(300)
with open('scan.log') as f:
 print(f.read()[-2000:])


[ 377.0s] done 24/81 (+30 brackets, total 736)
[ 490.5s] done 25/81 (+31 brackets, total 767)
[ 490.9s] done 26/81 (+32 brackets, total 799)
[ 491.4s] done 27/81 (+31 brackets, total 830)
[ 494.4s] done 28/81 (+31 brackets, total 861)
[ 494.4s] done 29/81 (+31 brackets, total 892)
[ 495.5s] done 30/81 (+31 brackets, total 923)
[ 495.8s] done 31/81 (+32 brackets, total 955)
[ 498.5s] done 32/81 (+31 brackets, total 986)
[ 610.5s] done 33/81 (+32 brackets, total 1018)
[ 611.1s] done 34/81 (+31 brackets, total 1049)
[ 612.4s] done 35/81 (+31 brackets, total 1080)
[ 615.7s] done 36/81 (+32 brackets, total 1112)
[ 616.3s] done 37/81 (+32 brackets, total 1144)
[ 616.4s] done 38/81 (+31 brackets, total 1175)
[ 616.6s] done 39/81 (+31 brackets, total 1206)
[ 619.2s] done 40/81 (+31 brackets, total 1237)
[ 731.9s] done 41/81 (+32 brackets, total 1269)
[ 753.3s] done 42/81 (+31 brackets, total 1300)
[ 824.8s] done 43/81 (+32 brackets, total 1332)
[ 837.5s] done 44/81 (+32 brackets, total 1364)
[

In [39]:
# 64/81 at 1285s. Linear extrapolation: 81/64 * 1285 = 1626s. Almost done.
# Wait some more.
import time
time.sleep(300)
with open('scan.log') as f:
 content = f.read()
print(content[-1800:])


one 39/81 (+31 brackets, total 1206)
[ 619.2s] done 40/81 (+31 brackets, total 1237)
[ 731.9s] done 41/81 (+32 brackets, total 1269)
[ 753.3s] done 42/81 (+31 brackets, total 1300)
[ 824.8s] done 43/81 (+32 brackets, total 1332)
[ 837.5s] done 44/81 (+32 brackets, total 1364)
[ 838.2s] done 45/81 (+32 brackets, total 1396)
[ 838.6s] done 46/81 (+31 brackets, total 1427)
[ 839.0s] done 47/81 (+32 brackets, total 1459)
[ 840.9s] done 48/81 (+32 brackets, total 1491)
[ 952.4s] done 49/81 (+32 brackets, total 1523)
[ 976.5s] done 50/81 (+31 brackets, total 1554)
[ 1046.2s] done 51/81 (+33 brackets, total 1587)
[ 1056.7s] done 52/81 (+31 brackets, total 1618)
[ 1058.2s] done 53/81 (+32 brackets, total 1650)
[ 1060.2s] done 54/81 (+32 brackets, total 1682)
[ 1060.2s] done 55/81 (+32 brackets, total 1714)
[ 1062.8s] done 56/81 (+32 brackets, total 1746)
[ 1174.7s] done 57/81 (+32 brackets, total 1778)
[ 1198.0s] done 58/81 (+32 brackets, total 1810)
[ 1268.2s] done 59/81 (+32 brackets, total 

In [40]:
# Note 1621.8s: 74/81 with +0 brackets! That's suspicious. Possibly chunk with 0 sign-changes (unlikely at high t). Maybe an error.
# We have 2325 brackets so far from scan. Total expected ~2400 from new scan + 2622 from old = ~5022. Good.
# Almost done. Let's keep waiting.
import time
time.sleep(180)
with open('scan.log') as f:
 print(f.read()[-1800:])


/81 (+31 brackets, total 1427)
[ 839.0s] done 47/81 (+32 brackets, total 1459)
[ 840.9s] done 48/81 (+32 brackets, total 1491)
[ 952.4s] done 49/81 (+32 brackets, total 1523)
[ 976.5s] done 50/81 (+31 brackets, total 1554)
[ 1046.2s] done 51/81 (+33 brackets, total 1587)
[ 1056.7s] done 52/81 (+31 brackets, total 1618)
[ 1058.2s] done 53/81 (+32 brackets, total 1650)
[ 1060.2s] done 54/81 (+32 brackets, total 1682)
[ 1060.2s] done 55/81 (+32 brackets, total 1714)
[ 1062.8s] done 56/81 (+32 brackets, total 1746)
[ 1174.7s] done 57/81 (+32 brackets, total 1778)
[ 1198.0s] done 58/81 (+32 brackets, total 1810)
[ 1268.2s] done 59/81 (+32 brackets, total 1842)
[ 1276.8s] done 60/81 (+32 brackets, total 1874)
[ 1279.8s] done 61/81 (+32 brackets, total 1906)
[ 1282.0s] done 62/81 (+32 brackets, total 1938)
[ 1282.4s] done 63/81 (+33 brackets, total 1971)
[ 1284.7s] done 64/81 (+32 brackets, total 2003)
[ 1395.0s] done 65/81 (+32 brackets, total 2035)
[ 1418.8s] done 66/81 (+32 brackets, total

In [41]:
# Scan complete! 2520 new brackets. Total = 2622 + 2520 = 5142.
# But brackets at 2499.4-2500 might be duplicated. Let me check.
import pickle
with open('brackets_0_2500.pkl','rb') as f:
 b1 = pickle.load(f)
with open('brackets_2500_4500.pkl','rb') as f:
 b2 = pickle.load(f)
print("b1:", len(b1), "last 3:", b1[-3:])
print("b2:", len(b2), "first 3:", b2[:3], "last 3:", b2[-3:])


b1: 2622 last 3: [(2498.299999999956, 2498.399999999956), (2498.6999999999557, 2498.7999999999556), (2499.399999999955, 2499.499999999955)]
b2: 2520 first 3: [(2499.4, 2499.5), (2500.1999999999994, 2500.2999999999993), (2501.0999999999985, 2501.1999999999985)] last 3: [(4497.500000000084, 4497.600000000084), (4498.0000000000855, 4498.100000000086), (4499.1000000000895, 4499.20000000009)]


In [42]:
# 1 overlap at 2499.4-2499.5. After dedup: 2622+2519 = 5141 brackets.
# That's MORE than 5000. We need to truncate to 5000 zeros.
# 
# Note: we'd need to go up to t ~ 4470 to get exactly 5000 zeros (N_approx(4470)~5108). Actually we should sort all zeros first.
# 
# Now refinement. Remaining time: 3600 - 30 (setup+init) - 1707 (scan) - misc = budget tight.
# Let's check current "used" time:

import os, time
# It was launched ~6 min in. Now we've been running quite a while. Let me check actual time elapsed since notebook start.
# Just check time of files
sb_stat = os.stat('brackets_2500_4500.pkl')
print("scan finished at:", time.ctime(sb_stat.st_mtime))
print("now:", time.ctime(time.time()))
print("time since scan finish:", time.time() - sb_stat.st_mtime, "s")


scan finished at: Sat May 30 19:54:37 2026
now: Sat May 30 19:57:06 2026
time since scan finish: 149.5521011352539 s


In [43]:
# Now decide refinement budget. We have used roughly 1850s so far. 1750s remaining.
# Refining 5000 zeros at dps=15 (~6 evals each at ~1s) = 30000s/8 = 3750s. Too slow.
# 
# Pragmatic plan: 
# 1. Run refinement with target_dps=15, maxiter=8 (which generally gives 12-15 digits)
# 2. Refine for ~1500s wall time, save partial results.
# 3. The partial results will cover at least the lower-t zeros (which are fastest).
# 
# Actually because we use imap (ordered), the partial results will be SORTED by t, so lower-t zeros refine first.
# 
# Use imap not imap_unordered → preserves order, low-t zeros done first, saves incrementally.
# 
# Limit wall time to ~1500s. Start now and let it run as long as we can.
# 
# Also: After saving brackets_2500_4500.pkl deduped properly:
all_b = b1 + b2[1:] # skip duplicate
all_b.sort(key=lambda ab: ab[0])
print("Total deduped brackets:", len(all_b))

# Sanity check: count expected zeros up to that t
print("Max t:", all_b[-1][1])
print("N_approx at max t:", (all_b[-1][1]/(2*3.14159265)) * (5*all_b[-1][1]/(2*3.14159265*2.71828)) ** 0 )
import math
T = all_b[-1][1]
print("N_approx:", T/(2*math.pi)*math.log(5*T/(2*math.pi*math.e)))


Total deduped brackets: 5141
Max t: 4499.20000000009
N_approx at max t: 716.069920777299
N_approx: 5143.684711541669


In [44]:
# 5141 brackets, N_approx ≈ 5144. Good match → all zeros captured in this range.
# Truncate to first 5000 for the target list.
all_b_5000 = all_b[:5000]
print("first bracket:", all_b_5000[0], "last bracket:", all_b_5000[-1])

# Save combined brackets
with open('brackets_all_5141.pkl', 'wb') as f:
 pickle.dump(all_b, f)
with open('brackets_first_5000.pkl', 'wb') as f:
 pickle.dump(all_b_5000, f)
print("Saved")


first bracket: (6.5999999999999925, 6.699999999999992) last bracket: (4390.500000000058, 4390.600000000059)
Saved


In [45]:
# Now write the refinement script - more focused.
refine_script = r'''
import sys, os, time, pickle
sys.path.insert(0, '/workspace/321f0eeb-097c-4d58-9349-fa9ad588875f')
os.chdir('/workspace/321f0eeb-097c-4d58-9349-fa9ad588875f')
import mpmath as mp
from lchi5_module import Z
from multiprocessing import Pool

def fast_refine_secant(args):
 a, b, target_dps, maxiter = args
 mp.mp.dps = target_dps + 5
 a = mp.mpf(a); b = mp.mpf(b)
 za = Z(a); zb = Z(b)
 eps = mp.mpf(10)**(-target_dps)
 c = (a+b)/2
 for _ in range(maxiter):
 if zb == za:
 c = (a+b)/2
 else:
 c = a - za*(b-a)/(zb-za)
 if not (a < c < b):
 c = (a+b)/2
 if (b - a) < eps * max(abs(c), mp.mpf(1)):
 break
 zc = Z(c)
 if abs(zc) < eps * mp.mpf(10):
 return mp.nstr(c, target_dps+2, strip_zeros=False)
 if (zc > 0) == (za > 0):
 a, za = c, zc
 else:
 b, zb = c, zc
 return mp.nstr(c, target_dps+2, strip_zeros=False)

def refine_chunk(args):
 brackets, target_dps, maxiter = args
 mp.mp.dps = target_dps + 5
 out = []
 for (a, b) in brackets:
 s = fast_refine_secant((a, b, target_dps, maxiter))
 out.append(s)
 return out

with open('brackets_first_5000.pkl','rb') as f:
 all_b = pickle.load(f)
print(f"Total brackets to refine: {len(all_b)}", flush=True)

TARGET_DPS = int(os.environ.get('TARGET_DPS', '15'))
MAXITER = int(os.environ.get('MAXITER', '7'))
CHUNK_SIZE = int(os.environ.get('CHUNK_SIZE', '25'))

# Build chunks
chunks = []
for ci in range(0, len(all_b), CHUNK_SIZE):
 bs = all_b[ci:ci+CHUNK_SIZE]
 chunks.append((bs, TARGET_DPS, MAXITER))

print(f"Chunks: {len(chunks)}, chunk_size={CHUNK_SIZE}, target_dps={TARGET_DPS}, maxiter={MAXITER}", flush=True)

t0 = time.time()
results = []
with Pool(8) as p:
 for i, res in enumerate(p.imap(refine_chunk, chunks)):
 results.append(res)
 elapsed = time.time()-t0
 flat = sum((len(r) for r in results), 0)
 print(f"[{elapsed:7.1f}s] chunk {i+1}/{len(chunks)} done; total refined: {flat}", flush=True)
 if i % 5 == 0 or i == len(chunks)-1:
 with open(f'zeros_partial_dps{TARGET_DPS}.pkl', 'wb') as f:
 pickle.dump([z for chunk in results for z in chunk], f)

flat_results = [z for chunk in results for z in chunk]
print(f"Total refined: {len(flat_results)} in {time.time()-t0:.1f}s")
flat_results.sort(key=lambda s: mp.mpf(s))
with open(f'zeros_Lchi5_N{len(flat_results)}_dps{TARGET_DPS}.pkl', 'wb') as f:
 pickle.dump(flat_results, f)
print(f"Saved zeros_Lchi5_N{len(flat_results)}_dps{TARGET_DPS}.pkl")
'''

with open('run_refine.py', 'w') as f:
 f.write(refine_script)
print("written")


written


In [46]:
# Launch refinement now with target_dps=15, maxiter=7 (gives ~10-13 digits)
import subprocess, time, os
env = os.environ.copy()
env['TARGET_DPS'] = '15'
env['MAXITER'] = '7'
env['CHUNK_SIZE'] = '25'
log = open('refine.log', 'w')
proc = subprocess.Popen(['python', 'run_refine.py'], stdout=log, stderr=subprocess.STDOUT, env=env)
print("PID:", proc.pid)
time.sleep(20)
with open('refine.log') as f:
 print(f.read())


PID: 60


Total brackets to refine: 5000
Chunks: 200, chunk_size=25, target_dps=15, maxiter=7
[ 4.4s] chunk 1/200 done; total refined: 25
[ 8.9s] chunk 2/200 done; total refined: 50
[ 10.0s] chunk 3/200 done; total refined: 75
[ 15.6s] chunk 4/200 done; total refined: 100
[ 16.1s] chunk 5/200 done; total refined: 125
[ 16.1s] chunk 6/200 done; total refined: 150
[ 16.6s] chunk 7/200 done; total refined: 175



In [47]:
# Going fast at low t! Let me wait a bit and see how scaling at higher t works.
import time
time.sleep(120)
with open('refine.log') as f:
 print(f.read()[-2000:])


Total brackets to refine: 5000
Chunks: 200, chunk_size=25, target_dps=15, maxiter=7
[ 4.4s] chunk 1/200 done; total refined: 25
[ 8.9s] chunk 2/200 done; total refined: 50
[ 10.0s] chunk 3/200 done; total refined: 75
[ 15.6s] chunk 4/200 done; total refined: 100
[ 16.1s] chunk 5/200 done; total refined: 125
[ 16.1s] chunk 6/200 done; total refined: 150
[ 16.6s] chunk 7/200 done; total refined: 175
[ 20.7s] chunk 8/200 done; total refined: 200
[ 29.7s] chunk 9/200 done; total refined: 225
[ 34.6s] chunk 10/200 done; total refined: 250
[ 36.7s] chunk 11/200 done; total refined: 275
[ 41.5s] chunk 12/200 done; total refined: 300
[ 42.9s] chunk 13/200 done; total refined: 325
[ 42.9s] chunk 14/200 done; total refined: 350
[ 42.9s] chunk 15/200 done; total refined: 375
[ 46.8s] chunk 16/200 done; total refined: 400
[ 57.6s] chunk 17/200 done; total refined: 425
[ 66.3s] chunk 18/200 done; total refined: 450
[ 81.4s] chunk 19/200 done; total refined: 475
[ 84.3s] chunk 20/200 done; total ref

In [48]:
# 33/200 chunks at 143s. Rate ~4.3 chunks/s wait. 200 chunks projected: 200/33 * 143 = 867s.
# But each chunk has 25 zeros, so 5000 zeros in ~867s. Quite optimistic for low-t.
# 
# But for HIGH-t chunks (t > 3000), the cost will dominate. Let me wait more.
import time
time.sleep(240)
with open('refine.log') as f:
 print(f.read()[-2500:])


done; total refined: 350
[ 42.9s] chunk 15/200 done; total refined: 375
[ 46.8s] chunk 16/200 done; total refined: 400
[ 57.6s] chunk 17/200 done; total refined: 425
[ 66.3s] chunk 18/200 done; total refined: 450
[ 81.4s] chunk 19/200 done; total refined: 475
[ 84.3s] chunk 20/200 done; total refined: 500
[ 85.1s] chunk 21/200 done; total refined: 525
[ 86.2s] chunk 22/200 done; total refined: 550
[ 86.4s] chunk 23/200 done; total refined: 575
[ 90.7s] chunk 24/200 done; total refined: 600
[ 99.1s] chunk 25/200 done; total refined: 625
[ 107.6s] chunk 26/200 done; total refined: 650
[ 121.9s] chunk 27/200 done; total refined: 675
[ 124.7s] chunk 28/200 done; total refined: 700
[ 127.0s] chunk 29/200 done; total refined: 725
[ 129.3s] chunk 30/200 done; total refined: 750
[ 129.5s] chunk 31/200 done; total refined: 775
[ 133.5s] chunk 32/200 done; total refined: 800
[ 143.0s] chunk 33/200 done; total refined: 825
[ 152.6s] chunk 34/200 done; total refined: 850
[ 164.4s] chunk 35/200 don

In [49]:
# We have ~1068s remaining. Currently at chunk 64/200 = 1600/5000 zeros refined in 395s.
# Refinement rate is slowing as t increases. Let's check projections.
# 
# Looking at the log: from chunk 40 (1000 zeros, 177s) to chunk 64 (1600 zeros, 395s):
# 600 zeros in 218s, rate ~2.75 zeros/s wall
# At higher t this will slow further. Best we can do is wait and save final results.
# 
# Decision: Let it run as long as possible, then take partial results.
# Reserve ~150s for final wrap-up.
import time
time.sleep(500)
with open('refine.log') as f:
 print(f.read()[-2500:])


TimeoutError: Code execution timed out after 449.0 seconds